# 🍷 Wine Quality Prediction System
**Data Mining Laboratory – Section B**  |  Python • GitHub • Streamlit • Predictive Modeling • SAD

**Name:** Baclay Christ John S.  **Section:** 3B  **Date:** November 9, 2026
### What you are building
A small tool that looks at a wine's lab measurements (acidity, sugar, alcohol, ...) and says whether it is
**GOOD QUALITY** or **NEEDS IMPROVEMENT**.

**Prediction rule:** quality ≥ 6 → GOOD QUALITY, quality < 6 → NEEDS IMPROVEMENT.
The original `quality` score is only the answer key (the *target*). It must **never** be used as an input.

> **Responsible-use note:** This is a classroom prototype on a public dataset. A prediction is not a substitute for laboratory testing, sensory evaluation, food-safety procedures, or professional quality-control decisions.


## Part 2 – Set up and load the dataset (15 min)

### Step 2.1 – Install the packages
**What this does:** installs the libraries we need (data handling, machine learning, web app, saving the model). Python 3.10+ is required (Colab already has it).

In [1]:
!pip install -q pandas scikit-learn streamlit joblib
import sys
print("Python version:", sys.version)

Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]


### Step 2.2 – Create the project folder
**What this does:** makes a folder called `wine-quality-prediction` and moves into it. All files created later are saved here.

In [2]:
import os
os.makedirs("wine-quality-prediction", exist_ok=True)
os.chdir("wine-quality-prediction")
print("Working folder:", os.getcwd())

Working folder: /content/wine-quality-prediction


### Step 2.3 – Provide the wine CSV (you already downloaded it)
**What this does:** looks for `winequality-red.csv` in the project folder. If it is not there (Colab), it asks you to upload the
**`winequality.zip` you downloaded** (or just the `winequality-red.csv` file). It unzips automatically.

- A window with a **Choose files** button will appear when you run the cell. Pick `winequality.zip`.
- The zip contains a red and a white wine file. **Use only the red wine file** – the lab is about red wine.
- Running on your own computer (VS Code)? Copy `winequality-red.csv` into the project folder first.

In [3]:
import os, zipfile

if not os.path.exists("winequality-red.csv"):
    try:
        from google.colab import files
        uploaded_files = files.upload()          # choose winequality.zip (or winequality-red.csv)
        for name in uploaded_files:
            if name.lower().endswith(".zip"):
                zipfile.ZipFile(name).extractall(".")
    except ImportError:
        print("Not in Colab: copy winequality-red.csv into this folder.")

if not os.path.exists("winequality-red.csv"):
    # the zip may have put the file inside a sub-folder - search for it
    for root, _, fs in os.walk("."):
        if "winequality-red.csv" in fs and root != ".":
            import shutil
            shutil.copy(os.path.join(root, "winequality-red.csv"), "winequality-red.csv")
            break

print("winequality-red.csv found:", os.path.exists("winequality-red.csv"))

Saving winequality-red.csv to winequality-red.csv
winequality-red.csv found: True


### Step 2.4 – Activity A: inspect the dataset
**What this does:** reads the CSV (it uses `;` between values), saves a clean copy called `wine_quality.csv`, and prints a quick health check:
size, first rows, data types, missing values, duplicate rows, and how many wines have each quality score.
(If the file is missing, it falls back to downloading from the UCI link in the lab sheet.)

**Dataset source:** https://archive.ics.uci.edu/dataset/186/wine+quality

In [4]:
import pandas as pd
import os

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
source = "winequality-red.csv" if os.path.exists("winequality-red.csv") else url
df = pd.read_csv(source, sep=";")
df.to_csv("wine_quality.csv", index=False)

print("Rows and columns:", df.shape)
print(df.head())
print(df.dtypes)
print("Missing values per column:")
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Quality distribution:")
print(df["quality"].value_counts().sort_index())

Rows and columns: (1599, 12)
   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0                  34.0   0.9978  3.51       0.56   

   alcohol  quality  
0      9.4        5  
1      9.8   

**Checkpoint ✅** – You should see **1599 rows and 12 columns**, **0 missing values**, **240 duplicate rows**,
quality scores from 3 to 8 (most wines are 5 or 6), and a file `wine_quality.csv` was created. The target column is named `quality`.

✏️ **Note down:** rows = ______ , columns = ______ , missing = ______ , duplicates = ______

## Part 3 – Preprocess, train, and evaluate (Activity B, 40 min)

### What happens in the code
1. **Remove duplicate rows** so the same wine is not counted twice.
2. **Create the target:** `1` if quality ≥ 6 (Good), `0` otherwise (Needs improvement).
3. **Drop `quality` from the inputs** (`X`). If we kept it, the model would just read the answer – this is called **target leakage**.
4. **Split the data:** 80% to train, 20% to test. `stratify=y` keeps the same good/bad proportion in both parts.
5. **Pipeline:** fill any missing numbers with the median → scale the numbers to the same range → train a **Random Forest** (150 decision trees).
6. **Evaluate** on the 20% the model has never seen, then **save** the model as `wine_model.pkl`.

### Step 3.1 – Write `train_model.py`
**What this does:** saves the complete training script (Activity A + B, exactly as in the lab sheet) as a file you will submit and push to GitHub.
The only small change: it reads your local `winequality-red.csv` first and uses the UCI link only as a backup.

In [5]:
%%writefile train_model.py
# train_model.py - Activity A + B
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix
)
import joblib

# ---------- Activity A: retrieve and inspect ----------
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
source = "winequality-red.csv" if os.path.exists("winequality-red.csv") else url
df = pd.read_csv(source, sep=";")
df.to_csv("wine_quality.csv", index=False)

print("Rows and columns:", df.shape)
print(df.head())
print(df.dtypes)
print("Missing values per column:")
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Quality distribution:")
print(df["quality"].value_counts().sort_index())

# ---------- Activity B: preprocess, train, evaluate ----------
# Remove duplicate records and define the binary target
df = df.drop_duplicates().copy()
y = (df["quality"] >= 6).astype(int)  # 1 = Good; 0 = Needs improvement

# Do not include the original quality score as a predictor
X = df.drop(columns=["quality"])

# Split the data while preserving class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Preprocessing and classification in one pipeline
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(
        n_estimators=150,
        random_state=42,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, predictions), 3))
print("Confusion matrix:")
print(confusion_matrix(y_test, predictions))
print(classification_report(
    y_test, predictions,
    target_names=["Needs improvement", "Good quality"],
    zero_division=0
))

joblib.dump(model, "wine_model.pkl")
print("Saved wine_model.pkl")

Writing train_model.py


### Step 3.2 – Run the script
**What this does:** runs `train_model.py`. It trains the model and prints the results. Takes a few seconds.

In [6]:
!python train_model.py

Rows and columns: (1599, 12)
   fixed acidity  volatile acidity  citric acid  ...  sulphates  alcohol  quality
0            7.4              0.70         0.00  ...       0.56      9.4        5
1            7.8              0.88         0.00  ...       0.68      9.8        5
2            7.8              0.76         0.04  ...       0.65      9.8        5
3           11.2              0.28         0.56  ...       0.58      9.8        6
4            7.4              0.70         0.00  ...       0.56      9.4        5

[5 rows x 12 columns]
fixed acidity           float64
volatile acidity        float64
citric acid             float64
residual sugar          float64
chlorides               float64
free sulfur dioxide     float64
total sulfur dioxide    float64
density                 float64
pH                      float64
sulphates               float64
alcohol                 float64
quality                   int64
dtype: object
Missing values per column:
fixed acidity           0
volat

**Checkpoint ✅** – `wine_model.pkl` now exists.

### Step 3.3 – Evaluation summary (50–65 min)
**What this does:** reloads the saved model, tests it again on the same 20% test set, and shows a clean table plus the confusion matrix. It also saves everything to `evaluation_results.txt` so you can copy the numbers into your report.

**How to read the numbers**
- **Accuracy** – share of all wines classified correctly (do **not** judge the model by this alone).
- **Precision (Good)** – when the model says "GOOD", how often is it right?
- **Recall (Good)** – of all truly good wines, how many did the model find?
- **F1-score** – a balance of precision and recall.
- **Confusion matrix** – rows = actual, columns = predicted: `[[true Needs-improvement, wrongly called Good], [wrongly called Needs-improvement, true Good]]`.

In [7]:
import pandas as pd, joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv("wine_quality.csv").drop_duplicates().copy()
y = (df["quality"] >= 6).astype(int)
X = df.drop(columns=["quality"])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
model = joblib.load("wine_model.pkl")
pred = model.predict(X_test)

acc = accuracy_score(y_test, pred)
rep = classification_report(
    y_test, pred, target_names=["Needs improvement", "Good quality"],
    zero_division=0, output_dict=True
)
table = pd.DataFrame({
    "Class": ["Needs improvement", "Good quality"],
    "Precision": [rep["Needs improvement"]["precision"], rep["Good quality"]["precision"]],
    "Recall": [rep["Needs improvement"]["recall"], rep["Good quality"]["recall"]],
    "F1-score": [rep["Needs improvement"]["f1-score"], rep["Good quality"]["f1-score"]],
    "Support": [rep["Needs improvement"]["support"], rep["Good quality"]["support"]],
}).round(3)

print("Accuracy:", round(acc, 3))
print("Test samples:", len(y_test))
print("\nConfusion matrix (rows = actual, cols = predicted):")
print(confusion_matrix(y_test, pred))
print()
print(table.to_string(index=False))

with open("evaluation_results.txt", "w") as f:
    f.write(f"Accuracy: {acc:.3f}\n\n")
    f.write(table.to_string(index=False))
print("\nSaved evaluation_results.txt")

Accuracy: 0.772
Test samples: 272

Confusion matrix (rows = actual, cols = predicted):
[[ 99  29]
 [ 33 111]]

            Class  Precision  Recall  F1-score  Support
Needs improvement      0.750   0.773     0.762    128.0
     Good quality      0.793   0.771     0.782    144.0

Saved evaluation_results.txt


✏️ **Write your results** (reference run: accuracy ≈ 0.77, about 0.74–0.80 for each class; small differences are normal):

| | Precision | Recall | F1-score |
|---|---|---|---|
| Needs improvement | | | |
| Good quality | | | |

**Accuracy:** ______

✏️ **Interpretation (2–3 sentences):** Which class is predicted better? Where are the most mistakes? Why is accuracy not enough?

## Part 4 – Build the Streamlit app (Activity C, 40 min)

**Streamlit** turns a Python file into a web page. `app.py` will:
1. show dataset facts (records, average quality, number of good wines) and a preview,
2. offer a **sample CSV** to download,
3. let you **upload a CSV**, check that all required columns are present, and show an error if not,
4. show the **prediction for each row** and let you **download the results**.

### Step 4.1 – Create `requirements.txt`
**What this does:** lists the packages so anyone can install them with `pip install -r requirements.txt`.

In [8]:
%%writefile requirements.txt
pandas
scikit-learn
streamlit
joblib

Writing requirements.txt


### Step 4.2 – Create `app.py`
**What this does:** saves the Streamlit app code from the lab sheet. (I only added a short responsible-use message at the bottom.)

In [9]:
%%writefile app.py
# app.py
import streamlit as st
import pandas as pd
import joblib

st.set_page_config(
    page_title="Wine Quality Predictor",
    page_icon="🍷",
    layout="wide"
)

@st.cache_resource
def load_model():
    return joblib.load("wine_model.pkl")

model = load_model()
data = pd.read_csv("wine_quality.csv")

st.title("🍷 Wine Quality Prediction System")
st.caption("Section B classroom prototype for preliminary quality classification")

c1, c2, c3 = st.columns(3)
c1.metric("Records", len(data))
c2.metric("Average quality score", round(data["quality"].mean(), 2))
c3.metric("Good-quality records", int((data["quality"] >= 6).sum()))

st.subheader("Dataset preview")
st.dataframe(data.head(10), use_container_width=True)

st.subheader("Predict uploaded wine samples")
st.write("Upload a CSV containing the same predictor columns as the training data.")

template = data.drop(columns=["quality"]).head(5)
st.download_button(
    "Download sample input CSV",
    template.to_csv(index=False),
    "sample_wine_input.csv",
    "text/csv"
)

uploaded = st.file_uploader("Choose wine sample CSV", type=["csv"])

if uploaded is not None:
    new_data = pd.read_csv(uploaded)
    required_columns = list(model.feature_names_in_)
    missing = [c for c in required_columns if c not in new_data.columns]

    if missing:
        st.error("Missing required columns: " + ", ".join(missing))
    else:
        predictions = model.predict(new_data[required_columns])
        results = new_data.copy()
        results["Prediction"] = [
            "GOOD QUALITY" if value == 1 else "NEEDS IMPROVEMENT"
            for value in predictions
        ]

        st.subheader("Prediction results")
        st.dataframe(results, use_container_width=True)
        st.metric(
            "Predicted good-quality samples",
            int((predictions == 1).sum())
        )
        st.download_button(
            "Download prediction results",
            results.to_csv(index=False),
            "wine_quality_predictions.csv",
            "text/csv"
        )

st.info(
    "Responsible use: this is a classroom prototype. A prediction is not a substitute "
    "for laboratory testing, sensory evaluation, food-safety procedures, or professional "
    "quality-control decisions."
)

Writing app.py


### Step 4.3 – Run the app

**Option A – on your own computer (recommended, this is what the lab expects).** Open a terminal in the project folder (the one with `app.py`, `wine_model.pkl`, `wine_quality.csv`) and run:
```
streamlit run app.py
```
Open **http://localhost:8501**. If `streamlit` is not recognised, use `python -m streamlit run app.py`.

**Option B – inside Google Colab.** Colab has no "localhost" you can open, so use these two cells:
1. The first cell starts the app in the background and prints a number like `34.123.45.67` – **copy it** (it is the tunnel password).
2. The second cell prints a link ending in `loca.lt`. Open it, paste the number when asked, then click to continue.
Leave the second cell running while you use the app.

**Using the app (for your screenshots):**
1. Click **Download sample input CSV**.
2. Upload it with **Choose wine sample CSV** → predictions appear. 📸 *Screenshot the dashboard and the results.*
3. Click **Download prediction results**.
4. Upload `bad_input_missing_alcohol.csv` (created in Part 6) → you should see the red error message. 📸

In [10]:
# Colab only - start the app in the background and show the tunnel password
!pip -q install streamlit
!streamlit run app.py --server.port 8501 --server.headless true &>/content/streamlit_logs.txt &
!sleep 5
print("Tunnel password (your public IP):")
!curl -s ipv4.icanhazip.com

Tunnel password (your public IP):
34.81.179.28


In [ ]:
# Colab only - create the public link (leave this cell running)
!npm install -s localtunnel
!npx localtunnel --port 8501

⠙your url is: https://big-shoes-kiss.loca.lt


## Part 5 – Publish to GitHub (Activity D, 20 min)

### Step 5.1 – Create `README.md`
**What this does:** writes the project description GitHub shows on the front page. It states the problem, dataset, prediction rule, technologies, how to run it, and the limitations (all required by the lab).

In [ ]:
%%writefile README.md
# 🍷 Wine Quality Prediction System

## Problem
A small wine-production business needs a consistent, preliminary way to screen wine samples using recorded physicochemical measurements. This classroom prototype classifies a sample as **GOOD QUALITY** or **NEEDS IMPROVEMENT**.

## Dataset
UCI Machine Learning Repository – Wine Quality (red wine):
https://archive.ics.uci.edu/dataset/186/wine+quality

## Prediction rule
- `quality >= 6` → GOOD QUALITY (1)
- `quality < 6` → NEEDS IMPROVEMENT (0)

The original `quality` score is used only to create the target. It is **not** used as an input feature (to avoid target leakage).

## Technologies
Python 3.10+, pandas, scikit-learn (Random Forest pipeline), joblib, Streamlit, GitHub.

## Installation and run steps
```bash
git clone https://github.com/YOUR-USERNAME/wine-quality-prediction.git
cd wine-quality-prediction
pip install -r requirements.txt
python train_model.py      # reads/downloads the red-wine CSV, trains the model, creates wine_quality.csv and wine_model.pkl
streamlit run app.py       # opens http://localhost:8501
```

## How to use the app
1. Click **Download sample input CSV**.
2. Upload it with **Choose wine sample CSV**.
3. View the predictions and click **Download prediction results**.

## Limitations
- Trained on one red-wine dataset; may not generalize to other wine types, producers, regions, or testing conditions.
- Predictions are preliminary screening only and are not a substitute for laboratory testing, sensory evaluation, food-safety procedures, or professional quality-control decisions.
- Quality scores are subjective sensory ratings, and the dataset is limited in size.

### Step 5.2 – Download your project files
**What this does:** zips the project folder and downloads it to your computer (Colab). Unzip it – you need `train_model.py`, `app.py`, `requirements.txt`, `README.md` (and optionally the CSV/model if your instructor allows).

In [ ]:
import shutil, os
os.chdir("..")
shutil.make_archive("wine-quality-prediction", "zip", "wine-quality-prediction")
try:
    from google.colab import files
    files.download("wine-quality-prediction.zip")
except ImportError:
    print("Not in Colab - the zip is in the current folder.")
os.chdir("wine-quality-prediction")

### Step 5.3 – Push to GitHub
1. Sign in at **github.com** → **New repository** → name it exactly `wine-quality-prediction` → **Public** → *Create repository*.
2. Open a terminal **in the folder with your files** (VS Code: Terminal → New Terminal) and run these commands.
   Replace `YOUR-USERNAME` with your GitHub username.
```bash
git init
git add train_model.py app.py requirements.txt README.md
git commit -m "Build wine quality prediction prototype"
git branch -M main
git remote add origin https://github.com/YOUR-USERNAME/wine-quality-prediction.git
git push -u origin main
```
3. Sign in with the approved method when asked (browser login / GitHub Desktop / VS Code sign-in). **Never** put passwords or tokens in your code.
4. Refresh the repository page: the 4 files and the README should be visible. 📸

✏️ **Repository URL:** `https://github.com/YOUR-USERNAME/wine-quality-prediction`

## Part 6 – Testing and reflection (20 min)

### Step 6.1 – Make the test files
**What this does:** creates (1) `sample_wine_input.csv` – the same file the app offers to download, and (2) `bad_input_missing_alcohol.csv` – a file with the `alcohol` column removed, to test the error message.

In [ ]:
import pandas as pd
data = pd.read_csv("wine_quality.csv")

data.drop(columns=["quality"]).head(5).to_csv("sample_wine_input.csv", index=False)
data.drop(columns=["quality", "alcohol"]).head(5).to_csv("bad_input_missing_alcohol.csv", index=False)

print("Created: sample_wine_input.csv, bad_input_missing_alcohol.csv")

### Step 6.2 – Quick check of the prediction logic (without the browser)
**What this does:** runs the same steps the app runs – check required columns, predict, label – so you can see the output right here.

In [ ]:
import pandas as pd, joblib
model = joblib.load("wine_model.pkl")

# Test 1: a good file -> predictions
sample = pd.read_csv("sample_wine_input.csv")
preds = model.predict(sample[list(model.feature_names_in_)])
out = sample.copy()
out["Prediction"] = ["GOOD QUALITY" if v == 1 else "NEEDS IMPROVEMENT" for v in preds]
print(out[["alcohol", "volatile acidity", "sulphates", "Prediction"]].to_string(index=False))

# Test 2: a file with a missing column -> helpful error
bad = pd.read_csv("bad_input_missing_alcohol.csv")
missing = [c for c in model.feature_names_in_ if c not in bad.columns]
print()
print("Missing-column test ->", "Missing required columns: " + ", ".join(missing))

### Step 6.3 – Test table
Tick ☑ each test that works in **your** run.

| Test | Expected result | Pass? |
|---|---|---|
| Run `train_model.py` | CSV and model files are created without errors | ☐ |
| Run Streamlit app | Dashboard loads in browser | ☐ |
| Upload sample CSV | Predictions are displayed | ☐ |
| Upload CSV missing a required column | The app displays a helpful error message | ☐ |
| Download results | A CSV containing predictions is downloaded | ☐ |
| Open GitHub repository | Source files and README are visible | ☐ |

### Step 6.4 – Reflection answers
*These are draft answers. Rewrite them in your own words and use your own numbers.*

**1. What is the prediction target, and why is the original quality column excluded from the input features?**
The target is a yes/no label made from the quality score: 1 (GOOD QUALITY) if quality ≥ 6, otherwise 0 (NEEDS IMPROVEMENT). The `quality` column is excluded because the target is calculated from it. Keeping it would be target leakage: the model would just read the answer, look almost perfect in testing, and fail in real use because a new sample has no quality score yet.

**2. What does recall for the GOOD QUALITY class tell you? What could low recall mean for the business?**
Recall for GOOD QUALITY is the share of truly good wines that the model correctly finds (true positives ÷ all actual good wines). Low recall means many good batches are wrongly flagged as NEEDS IMPROVEMENT, so the business may waste time and money re-testing or reworking good wine, or hold back wine that could be sold.

**3. Why might a model trained on this red-wine dataset perform differently on another producer's wine samples?**
The data covers one wine type (red, from a particular region and producers) rated by particular tasters. Other producers may use different grapes, processes, equipment and measurement methods, so the measurement ranges and the link between chemistry and quality can differ (data shift). The model may be less accurate on them.

**4. Why should the prediction be treated as a preliminary screening result rather than a final quality decision?**
The model is statistical and not perfectly accurate (it makes false positives and false negatives), and it only sees physicochemical measurements, not taste, smell or food-safety checks. Quality scores are also partly subjective. It can help decide which batches to look at first, but the final decision needs laboratory testing, sensory evaluation and professional quality control.

**5. What additional data or testing would help improve the reliability of the system?**
More samples (especially very low and very high quality), data from several producers, regions, vintages and wine types, and ratings from more tasters. Technically: cross-validation, comparing other models (Logistic Regression, Decision Tree), tuning the GOOD-QUALITY threshold, feature-importance analysis, and testing on an outside dataset. A real-world pilot against expert sensory and laboratory results would also help.

## Final submission checklist
- ☐ SAD mini-analysis (problem, user, inputs, process, outputs, limitation) – Part 1
- ☐ `train_model.py` and `app.py`
- ☐ `requirements.txt` and `README.md`
- ☐ Model evaluation results (accuracy, precision, recall, F1-score) – Part 3
- ☐ Screenshot of the Streamlit dashboard and prediction results
- ☐ GitHub repository URL
- ☐ Brief answers to the reflection questions – Part 6

**Optional extensions (outside the 150 minutes):** compare with Logistic Regression or Decision Tree, add a confusion-matrix plot and feature importances, use cross-validation, deploy on Streamlit Community Cloud, try a different GOOD-QUALITY threshold.